# Preliminari

Si impostano directory di lavoro e si fanno import per spark

In [18]:
import os
from pyspark.sql import SparkSession

DATASETS_DIR = "../dataset/"

spark = (
    SparkSession.builder
    .appName("pfp")
    .getOrCreate()
)

sc = spark.sparkContext


## Pre-Processing


Creo un dataframe dal file .parquet di input.

Creo i record (rdd) come necessario dal problema, ovvero chiave dell'ordine e valore le tuple contenente id oggetto e quantità.
Questi record li chiameremo Transazioni, come suggerito dal paper PFP.

In [19]:
sdf = spark.read.parquet(
    os.path.join(DATASETS_DIR, "online_retail.parquet")
)

transactions = (
    sdf
    .select("InvoiceNo", "StockCode", "Quantity")
    .rdd
    .map(lambda row: (row["InvoiceNo"], (row["StockCode"], row["Quantity"])))
    .groupByKey()
    .mapValues(list)
)
transactions.take(5)

[('536365',
  [('85123A', '6'),
   ('71053', '6'),
   ('84406B', '8'),
   ('84029G', '6'),
   ('84029E', '6'),
   ('22752', '2'),
   ('21730', '6')]),
 ('536366', [('22633', '6'), ('22632', '6')]),
 ('536367',
  [('84879', '32'),
   ('22745', '6'),
   ('22748', '6'),
   ('22749', '8'),
   ('22310', '6'),
   ('84969', '6'),
   ('22623', '3'),
   ('22622', '2'),
   ('21754', '3'),
   ('21755', '3'),
   ('21777', '4'),
   ('48187', '4')]),
 ('536368', [('22960', '6'), ('22913', '3'), ('22912', '3'), ('22914', '3')]),
 ('536369', [('21756', '3')])]

### Conversione

convertiamo gli oggetti (tuple chiave e quantità) in nuovi oggetti identificati da un numero, in questo modo si può facilmente utilizzare PFP con le quantità.
Riduciamo funzionalmente il problema di tenere in considerazione le quantità al problema senza le quantità per poi tornare al problema delle quantità.

T' = T
for t in T'
    t -> t'

out = PFP(T')

reversed = revert(out)

return alpha_code(reversed)

Per fare tutto ciò innanzitutto devo prendere gli oggetti e quantità e mapparli:

In [20]:
pairs= (
    transactions
    .flatMap(lambda x: x[1])                 # prendo tutte le tuple
    .distinct()                              # tuple uniche
    .sortBy(lambda pair: (pair[0], pair[1])) # ordine stabile
    .zipWithIndex()                          # assegna indice 0,1,2...
)

print("ci sono " + str(pairs.count()) + " coppie")
pairs.take(50)


ci sono 45280 coppie


[(('10002', '-3'), 0),
 (('10002', '1'), 1),
 (('10002', '10'), 2),
 (('10002', '11'), 3),
 (('10002', '12'), 4),
 (('10002', '120'), 5),
 (('10002', '14'), 6),
 (('10002', '18'), 7),
 (('10002', '180'), 8),
 (('10002', '2'), 9),
 (('10002', '24'), 10),
 (('10002', '3'), 11),
 (('10002', '36'), 12),
 (('10002', '4'), 13),
 (('10002', '48'), 14),
 (('10002', '5'), 15),
 (('10002', '6'), 16),
 (('10002', '60'), 17),
 (('10002', '62'), 18),
 (('10002', '8'), 19),
 (('10080', '1'), 20),
 (('10080', '12'), 21),
 (('10080', '170'), 22),
 (('10080', '2'), 23),
 (('10080', '22'), 24),
 (('10080', '24'), 25),
 (('10080', '26'), 26),
 (('10080', '3'), 27),
 (('10080', '4'), 28),
 (('10080', '48'), 29),
 (('10120', '1'), 30),
 (('10120', '10'), 31),
 (('10120', '11'), 32),
 (('10120', '12'), 33),
 (('10120', '2'), 34),
 (('10120', '20'), 35),
 (('10120', '3'), 36),
 (('10120', '30'), 37),
 (('10120', '4'), 38),
 (('10120', '5'), 39),
 (('10120', '6'), 40),
 (('10120', '8'), 41),
 (('10123C', '-18

In [21]:
# Creo la mappa di conversione da tupla a numero
conversion_map = pairs.collectAsMap()
# Lo distribuisco ai worker in broadcast
bc_map = sc.broadcast(conversion_map)
bc_map.value

{('10002', '-3'): 0,
 ('10002', '1'): 1,
 ('10002', '10'): 2,
 ('10002', '11'): 3,
 ('10002', '12'): 4,
 ('10002', '120'): 5,
 ('10002', '14'): 6,
 ('10002', '18'): 7,
 ('10002', '180'): 8,
 ('10002', '2'): 9,
 ('10002', '24'): 10,
 ('10002', '3'): 11,
 ('10002', '36'): 12,
 ('10002', '4'): 13,
 ('10002', '48'): 14,
 ('10002', '5'): 15,
 ('10002', '6'): 16,
 ('10002', '60'): 17,
 ('10002', '62'): 18,
 ('10002', '8'): 19,
 ('10080', '1'): 20,
 ('10080', '12'): 21,
 ('10080', '170'): 22,
 ('10080', '2'): 23,
 ('10080', '22'): 24,
 ('10080', '24'): 25,
 ('10080', '26'): 26,
 ('10080', '3'): 27,
 ('10080', '4'): 28,
 ('10080', '48'): 29,
 ('10120', '1'): 30,
 ('10120', '10'): 31,
 ('10120', '11'): 32,
 ('10120', '12'): 33,
 ('10120', '2'): 34,
 ('10120', '20'): 35,
 ('10120', '3'): 36,
 ('10120', '30'): 37,
 ('10120', '4'): 38,
 ('10120', '5'): 39,
 ('10120', '6'): 40,
 ('10120', '8'): 41,
 ('10123C', '-18'): 42,
 ('10123C', '1'): 43,
 ('10123C', '3'): 44,
 ('10123G', '-38'): 45,
 ('10124A

In [22]:
# Rimappo ogni transazione
transactions_ids = transactions.mapValues(
    lambda items: [bc_map.value[item] for item in items]
)
print(transactions_ids.count())
transactions_ids.take(5)

25900


[('536365', [42870, 36543, 38856, 38274, 38241, 23016, 9284]),
 ('536366', [21177, 21147]),
 ('536367',
  [40565,
   22913,
   22958,
   22975,
   16063,
   41170,
   20967,
   20951,
   9513,
   9530,
   9631,
   36264]),
 ('536368', [25906, 25161, 25151, 25171]),
 ('536369', [9543])]

## PFP

Iniziamo ad implementare **PFP**, definiamo una variabile **epsilon** che rappresenta la "predefined minimum support threshold"
soglia minima predefinita di supporto.
Quindi una threshold sopra la quale verrà riconosciuto un pattern e i pattern sotto questa soglia verranno scartati 

In [23]:
epsilon = 100

#supporto(item) = numero di transazioni che contengono lo stesso item

item_counts = (
    transactions_ids
    .flatMap(lambda row: set(row[1]))   # ogni item contato una sola volta per transazione
    .map(lambda item: (item, 1))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda row: row[1] >= epsilon)
)
print(item_counts.count())
item_counts.take(10)


970


[(23016, 121),
 (42870, 560),
 (22913, 138),
 (9513, 269),
 (22958, 138),
 (20951, 121),
 (9530, 201),
 (25906, 451),
 (9543, 103),
 (45258, 197)]

Creo la F-List che è la lista decrescente degli item (item = id_of(tuple(code, quantity)))

Poi ordino le transazione per "supporto" ovvero in base ai valori di F-List

Creo infine la Q-List tramite la quale si suddivide il calcolo tra le macchine.

In [24]:
# creo f_list
f_list = item_counts.sortBy(
    lambda row: (row[1], row[0]),
    ascending=False
)

f_list.take(10)

[(42543, 724),
 (45185, 708),
 (1994, 631),
 (40577, 597),
 (17796, 596),
 (5317, 583),
 (22436, 571),
 (42870, 560),
 (29325, 551),
 (29508, 513)]

In [25]:
# Creo una mappa per ordinare le transazioni, la mappa è fatta così:  item_id -> posizione nella F-list

# Base comune: item_id -> rank nella F-list
item_rank = (
    f_list
    .map(lambda row: row[0])      # item_id
    .zipWithIndex()               # item_id -> rank
    .map(lambda x: (x[0], int(x[1])))
    .persist()
)

f_rank = item_rank.collectAsMap()
# notifico i worker
bc_f_rank = sc.broadcast(f_rank)

ordered_transactions = (
    transactions_ids
    .mapValues(
        lambda items: sorted(
            # tieni item solo se item è una chiave del dizionario f_rank
            set(item for item in items if item in bc_f_rank.value),
            key=lambda item: bc_f_rank.value[item]
        )
    )
    .filter(lambda row: len(row[1]) > 0)
)
print(ordered_transactions.count())
print(ordered_transactions.take(5))


16578
[('536365', [42870, 23016]), ('536367', [9513, 9530, 22958, 22913, 20951]), ('536368', [25906]), ('536369', [9543]), ('536370', [18812, 45258])]


In [26]:
# G-List
Q = spark.sparkContext.defaultParallelism # numero di core disponibili tra tutti i worker

g_list = (
    item_rank
    .map(lambda x: (x[0], int(x[1] % Q)))   # item_id -> gid
    .collectAsMap()
)

bc_g_list = sc.broadcast(g_list)

In [27]:
#Questo è fondamentalmente il mapper del paper
def generate_group_dependent_transactions(row):
    invoice_no, items = row

    output = []
    seen_gids = set()

    # Scorro la transazione da destra verso sinistra
    for j in range(len(items) - 1, -1, -1):
        item = items[j]
        gid = bc_g_list.value.get(item)

        # Se questo gruppo non è ancora stato emesso per questa transazione
        if gid is not None and gid not in seen_gids:
            seen_gids.add(gid)

            # Emetto il prefisso fino alla posizione j inclusa
            output.append((gid, items[:j + 1]))

    return output

group_dependent_transactions = ordered_transactions.flatMap(
    generate_group_dependent_transactions
)

group_dependent_transactions.take(10)

# a ogni gid associo le transazioni di cui si deve occupare.

group_shards = group_dependent_transactions.groupByKey()

### Nodi e Alberi

A questo punto abbiamo bisogno degli FP-tree per minare i pattern. 
Ogni worker/reducer costruirà un FP-tree locale a partire dalle transazioni associate a uno specifico gruppo.

Dato che gli item di ogni transazione sono già ordinati secondo la F-list, e dato che le transazioni group-dependent sono raggruppate per `gid`, ogni gruppo può essere minato in modo indipendente dagli altri (come dimostrato nel paper).

Una transazione originale può generare più transazioni parziali, una per ogni gruppo presente nella transazione. Durante lo shuffle, questi prefissi vengono inviati ai reducer corrispondenti ai rispettivi gruppi.

Quindi una stessa transazione originale può contribuire a più FP-tree locali, ma ogni FP-tree locale contiene solo il sotto-database necessario per minare i pattern che terminano negli item del proprio gruppo.

In [28]:

from collections import defaultdict

# Creiamo una struttura di nodi in grado di navigare al parent e ai child.
class FPNode:
    def __init__(self, item=None, parent=None):
        self.item = item
        self.count = 0
        self.parent = parent
        self.children = {}

    def add_child(self, item):
        child = FPNode(item=item, parent=self)
        self.children[item] = child
        return child


# L'inserimento di una transazione può comportare la creazione di nodi figli oppure l'incremento del loro conteggio.

def insert_transaction(root, transaction, header_table, count=1):
    node = root
    for item in transaction:

        if item in node.children:
            child = node.children[item]
            child.count += count
        else:
            child = node.add_child(item)
            child.count = count
            header_table[item].append(child)

        node = child
    
# La header-table serve per avere una navigazione rapida ai nodi che rappresentano lo stesso item, infatti nell'albero possono 
# esserci N nodi che rappresentano lo stesso item, grazie alla header_table possiamo evitare di navigare tutto l'albero 
# ma abbiamo un accesso "diretto"

def build_fp_tree(transactions_iter):
    root = FPNode()
    header_table = defaultdict(list)

    for transaction in transactions_iter:
        insert_transaction(root, transaction, header_table, count=1)

    return root, dict(header_table)

def is_single_path(node):
    current = node

    while True:
        if len(current.children) == 0:
            return True
        if len(current.children) > 1:
            return False

        current = next(iter(current.children.values()))

        
def print_tree(node, indent=0, max_depth=3):
    if indent >= max_depth:
        return

    for child in node.children.values():
        print("  " * indent + f"{child.item}:{child.count}")
        print_tree(child, indent + 1, max_depth)



def count_nodes(node):
    total = 1
    for child in node.children.values():
        total += count_nodes(child)
    return total



def tree_stats_for_group(row):
    gid, transactions_iter = row

    root, header_table = build_fp_tree(transactions_iter)

    return (
        gid,
        count_nodes(root),
        len(header_table),
        list(root.children.keys())[:10]
    )

In [29]:
# Ora creiamo la nowGroup

gid_to_items_tmp = defaultdict(list)

for item_id, gid in g_list.items():
    gid_to_items_tmp[gid].append(item_id)
# gid -> item_ids
nowGroup = dict(gid_to_items_tmp)

bc_nowGroup = sc.broadcast(nowGroup)

In [30]:
#tree_stats = group_shards.map(tree_stats_for_group)
#gid, nodes_number, header_table_length,childern_length = tree_stats.take(1)[0]
#tree_stats.take(10)
#print("gid:", gid)
#print("item distinti nell'albero:", header_table_length)

In [31]:
# Dato un nodo, risale fino alla root e restituisce il cammino dei parent.
# Esempio se il nodo è d ed il path è a -> b -> c -> d restituisce [a,b,c] 
# Quindi non torna root e nodo corrente

def get_prefix_path(node):
    path = []
    current = node.parent
    while current is not None and current.item is not None:
        path.append(current.item)
        current = current.parent
    path.reverse()
    return path


# Per un item, prende tutti i nodi in header_table[item] e costruisce una lista degli elementi precedenti a lui nel path


def conditional_pattern_base(item, header_table):
    base = []
    for node in header_table.get(item, []):
        path = get_prefix_path(node)
        if path:
            base.append((path, node.count))
    return base


# conta gli item
# fa pruning degli item sotto soglia
# ordina i prefissi
# costruisce un nuovo FP-tree condizionato

def build_conditional_tree(pattern_base, min_support):
    # 1. conta item nella base pesando con node.count
    item_counts = defaultdict(int)
    for path, count in pattern_base:
        for item in path:
            item_counts[item] += count

    # 2. pruning
    frequent_items = {item for item, c in item_counts.items() if c >= min_support}
    if not frequent_items:
        return None, {}

    # 3. costruisco transazioni filtrate e ordinate
    root = FPNode()
    header_table = defaultdict(list)

    for path, count in pattern_base:
        filtered = [item for item in path if item in frequent_items]
        if filtered:
            insert_transaction(root, filtered, header_table, count=count)

    return root, dict(header_table)


# Per ogni item nell’albero:
# crea il pattern prefix + item
# salva il supporto
# costruisce il conditional tree
# richiama ricorsivamente mine_fp_tree

def mine_fp_tree(root, header_table, min_support, suffix=()):
    patterns = []

    items = sorted(
        header_table.keys(),
        key=lambda item: sum(node.count for node in header_table[item])
    )

    for item in items:
        support = sum(node.count for node in header_table[item])

        if support < min_support:
            continue

        new_pattern = tuple(sorted((item,) + suffix))
        patterns.append((new_pattern, support))

        base = conditional_pattern_base(item, header_table)
        cond_root, cond_header = build_conditional_tree(base, min_support)

        if cond_header:
            patterns.extend(
                mine_fp_tree(cond_root, cond_header, min_support, suffix=(item,) + suffix)
            )

    return patterns

In [32]:
#TESTING CODE BOX
gid, transactions_iter = group_shards.take(1)[0]
root, header_table = build_fp_tree(transactions_iter)

patterns = mine_fp_tree(root, header_table, min_support=epsilon)
patterns[:20]

[((17292,), 100),
 ((5594,), 100),
 ((25931,), 100),
 ((7064,), 100),
 ((13827,), 101),
 ((35975,), 101),
 ((2218,), 101),
 ((30762,), 101),
 ((41177,), 102),
 ((20900,), 102),
 ((19579,), 102),
 ((11445,), 102),
 ((38079,), 102),
 ((10968,), 102),
 ((29297,), 102),
 ((11896,), 103),
 ((18530,), 103),
 ((18426,), 103),
 ((2054,), 103),
 ((45259,), 103)]

In [33]:
def mine_group(row):
    gid, transactions_iter = row
    root, header_table = build_fp_tree(transactions_iter)
    return mine_fp_tree(root, header_table, min_support=100)

all_patterns = group_shards.flatMap(mine_group)

In [34]:
all_patterns.take(1000)

[((17292,), 100),
 ((5594,), 100),
 ((25931,), 100),
 ((7064,), 100),
 ((13827,), 101),
 ((35975,), 101),
 ((2218,), 101),
 ((30762,), 101),
 ((41177,), 102),
 ((20900,), 102),
 ((19579,), 102),
 ((11445,), 102),
 ((38079,), 102),
 ((10968,), 102),
 ((29297,), 102),
 ((11896,), 103),
 ((18530,), 103),
 ((18426,), 103),
 ((2054,), 103),
 ((45259,), 103),
 ((14951,), 103),
 ((17316,), 104),
 ((2989,), 105),
 ((15868,), 105),
 ((14555,), 105),
 ((26557,), 105),
 ((26495,), 105),
 ((41162,), 106),
 ((21689,), 106),
 ((7464,), 106),
 ((4508,), 106),
 ((25020,), 106),
 ((21618,), 106),
 ((16730,), 106),
 ((30938,), 106),
 ((19497,), 107),
 ((42855,), 107),
 ((5425,), 107),
 ((7459,), 107),
 ((11722,), 107),
 ((37742,), 107),
 ((12011,), 108),
 ((8587,), 108),
 ((5365,), 108),
 ((18291,), 108),
 ((22664,), 108),
 ((11758,), 108),
 ((24255,), 108),
 ((19731,), 108),
 ((11398,), 108),
 ((29844,), 108),
 ((17918,), 109),
 ((15954,), 109),
 ((11438,), 109),
 ((25063,), 109),
 ((22063,), 109),
 ((